In [165]:
# Import libraries
import pandas as pd
import numpy as np

In [166]:
# Load data
path = "D:/D/Projects/Healthcare Analytics/data/raw"

admissions = pd.read_csv(path+'/admissions.csv')
claims = pd.read_csv(path+'/claims.csv')
patients = pd.read_csv(path+'/patients.csv')
readmissions = pd.read_csv(path+'/readmissions.csv')

CREATE DATA QUALITY FLAGS

In [167]:
# Convert data column firt, try to do it first
admissions['admit_date'] = pd.to_datetime(admissions['admit_date'])
admissions['discharge_date'] = pd.to_datetime(admissions['discharge_date'])

In [168]:
# Start generate flag for los, dignosis, billing, claims
# Flag LOS data, and count negative day stay because it is impossible dtays
admissions['negative_los_flag'] = admissions['los_days']<0
print('Negative LOS count: ', (admissions['los_days']<0).sum())

# Date Logic error (litle different than negative los)
admissions['date_logic_error_flag'] = (admissions['discharge_date'] < admissions['admit_date'])
print(admissions['date_logic_error_flag'].sum())

# Try to check LOS mismatch
computed_los = (
    admissions['discharge_date'] - admissions['admit_date']
).dt.days

admissions['los_mismatch_flag'] = (
    admissions['los_days'] != computed_los
)
print("LOS mismatch count: ", admissions['los_mismatch_flag'].sum())

Negative LOS count:  2000
2000
LOS mismatch count:  3974


In [169]:
# missing diagnosis codes with ICD format check
admissions['diagnosis_missing_flag'] = (
    admissions['diagnosis_code'].isna()
)

# Perform standardization on diagnosis data to clea it. 
icd_pattern = r"^[A-Z][0-9]{2}(\.[A-Z0-9]{1,3})?$"

admissions['diagnosis_invalid_format_flag'] = (
    admissions['diagnosis_code']
    .astype("string")
    .str.strip()
    .str.upper()
    .str.match(icd_pattern, na=False)
)

# Suspicious diagnosis description
admissions['diagnosis_desc_invalid_flag'] = (
    admissions['diagnosis_desc']
    .astype("string")
    .str.strip()
    .str.upper()
    .isin(["INVALID", "N/A", "MISSING", ""])
)

In [170]:
# Orphan patient records
admissions['orphan_patient_flag'] = (
    ~admissions['patient_id'].isin(patients['patient_id'])
)

print("Orphan admissions count: ", admissions['orphan_patient_flag'].sum())

Orphan admissions count:  1000


In [171]:
# Final flagged data
admissions[
[
"negative_los_flag",
"date_logic_error_flag",
"los_mismatch_flag",
"diagnosis_missing_flag",
"diagnosis_invalid_format_flag",
"diagnosis_desc_invalid_flag",
"orphan_patient_flag"
]
].sum()

negative_los_flag                  2000
date_logic_error_flag              2000
los_mismatch_flag                  3974
diagnosis_missing_flag              512
diagnosis_invalid_format_flag    196488
diagnosis_desc_invalid_flag        4000
orphan_patient_flag                1000
dtype: Int64

In [172]:
# This enables:

# filtering safe datasets
# ML-ready datasets
# dashboard-safe subsets
# audit transparency
# reproducibility

# This is enterprise analytics thinking

FINANCIAL INTEGRITY VIOLATION

In [173]:
# Flag for billed amount and approve amount
claims['approved_exceeds_billed_flag'] = (
    claims['approved_amount'] > claims['billed_amount']
)
print("Approved Amount Flag Count: ", claims['approved_exceeds_billed_flag'].sum())

# Payment timeline violation
claims['submission_date'] = pd.to_datetime(claims['submission_date'])
claims['payment_date'] = pd.to_datetime(claims['payment_date'])

claims['payment_before_submission_flag'] = (
    claims['submission_date'] > claims['payment_date']
)
print("Paayment before claim submission count: ", claims['payment_before_submission_flag'].sum())


Approved Amount Flag Count:  4967
Paayment before claim submission count:  501


In [174]:
# Missing payment date when status is approved
claims['missing_payment_date_flag'] = (
    (claims['claim_status'] == "Approved") &
    (claims['payment_date'].isna())
)
print("Missing payment date for approved claims count: ", claims['missing_payment_date_flag'].sum())

# Payment exist but claim not approved
claims['payment_without_approval_flag'] = (
    claims["payment_date"].notna() &
    ~claims["claim_status"].isin(["Approved", "Partial"])
)
print("Payment Exist even claims pending count: ", claims['payment_without_approval_flag'].sum())

# Negative amount in billed
claims['negative_amount_flag'] = (
    (claims['billed_amount'] < 0) |
    (claims['approved_amount'] < 0)
)
print("Negative finacial count: ", claims['negative_amount_flag'].sum())

Missing payment date for approved claims count:  0
Payment Exist even claims pending count:  1966
Negative finacial count:  2000


In [175]:
# Claims flaged columns
claims[[
    "approved_exceeds_billed_flag",
    "payment_before_submission_flag",
    "negative_amount_flag",
    "missing_payment_date_flag",
    "payment_without_approval_flag"
]].sum()

approved_exceeds_billed_flag      4967
payment_before_submission_flag     501
negative_amount_flag              2000
missing_payment_date_flag            0
payment_without_approval_flag     1966
dtype: int64